In [1]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Newton-Raphson Power Flow

In this example, we are going to create a network with three nodes, three lines, one load, and one generator with given setpoints. We then solve the non-linear power flow equations using the Newton-Raphson method applied in `n.pf()`.

In [2]:
import numpy as np

import pypsa

n = pypsa.Network()

N_BUSES = 3

/tmp/ipykernel_3907/2652517220.py:5: PyPSA2Warning: pandas reads strings as `str` dtype since 3.0. PyPSA converts them back to object dtype on import, but keeps `str` from PyPSA 2.0 on. Set `pypsa.options.future.string_dtype` to True to opt in now or to False to keep the current behaviour.
  n = pypsa.Network()


Add three buses:

In [3]:
for i in range(N_BUSES):
    n.add("Bus", f"My bus {i}", v_nom=20)

n.buses

,v_nom,type,x,y,carrier,unit,scaling,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network
name,,,,,,,,,,,,,,
My bus 0,20.0,,0.0,0.0,AC,,1.0,,1.0,0.0,inf,PQ,,
My bus 1,20.0,,0.0,0.0,AC,,1.0,,1.0,0.0,inf,PQ,,
My bus 2,20.0,,0.0,0.0,AC,,1.0,,1.0,0.0,inf,PQ,,


Add three lines in a ring

In [4]:
for i in range(N_BUSES):
    n.add(
        "Line",
        f"My line {i}",
        bus0=f"My bus {i}",
        bus1=f"My bus {(i + 1) % N_BUSES}",
        x=0.1,
        r=0.01,
    )

n.lines

,bus0,bus1,type,x,r,g,b,s_nom,s_nom_mod,s_nom_extendable,...,v_ang_max,sub_network,x_pu,r_pu,g_pu,b_pu,x_pu_eff,r_pu_eff,s_nom_opt,capital_cost_piecewise_opt
name,,,,,,,,,,,,,,,,,,,,,
My line 0,My bus 0,My bus 1,,0.1,0.01,0.0,0.0,0.0,0.0,False,...,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
My line 1,My bus 1,My bus 2,,0.1,0.01,0.0,0.0,0.0,0.0,False,...,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
My line 2,My bus 2,My bus 0,,0.1,0.01,0.0,0.0,0.0,0.0,False,...,inf,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Add a generator at bus 0

In [5]:
n.add("Generator", "My gen", bus="My bus 0", p_set=100, control="PQ")

n.generators

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,min_down_time,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt,capital_cost_piecewise_opt
name,,,,,,,,,,,,,,,,,,,,,
My gen,My bus 0,PQ,,0.0,0.0,False,0.0,inf,NaN,0.0,...,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0,0.0


Add a load at bus 1

In [6]:
n.add("Load", "My load", bus="My bus 1", p_set=100, q_set=100)

n.loads

,bus,carrier,type,p_set,q_set,sign,active
name,,,,,,,
My load,My bus 1,,,100.0,100.0,-1.0,True


Do a Newton-Raphson power flow

In [7]:
n.pf()

INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x784b203ab0e0> for snapshots Index(['now'], dtype='str', name='snapshot')


/tmp/ipykernel_3907/470424023.py:1: PyPSA2Warning: From PyPSA 2.0 on, `get_switchable_as_dense` returns columns in `inds` order, keeps duplicates and raises a `KeyError` for unknown names. Unique, known names in component order give the same result in both. Set `pypsa.options.future.inds_order` to True to opt in now or to False to keep the current behaviour.
  n.pf()


{'n_iter': name      0
 snapshot   
 now       3,
 'error': name                 0
 snapshot              
 now       4.753531e-10,
 'converged': name         0
 snapshot      
 now       True}

Alright, it converged! Now, what is the active power flow on the lines?

In [8]:
n.lines_t.p0

name,My line 0,My line 1,My line 2
snapshot,,,
now,66.897487,-33.333333,-33.391038


...and what are the voltage angles on the buses?

In [9]:
n.buses_t.v_ang * 180 / np.pi

name,My bus 0,My bus 1,My bus 2
snapshot,,,
now,0.0,-0.875939,-0.433813


...and their mangitudes?

In [10]:
n.buses_t.v_mag_pu

name,My bus 0,My bus 1,My bus 2
snapshot,,,
now,1.0,0.981199,0.99057
